# 02 — Backpropagation from Scratch

In Notebook 01, we hardcoded the gradients for XOR. Now we **formalize** backprop:

1. **Computational graph** — every operation is a node with forward/backward
2. **Chain rule** — gradients flow backward through the graph automatically
3. **Gradient checking** — verify our math with numerical finite differences
4. **SGD optimizer** — the simplest optimizer, still the foundation of all modern ones

This is **exactly** what PyTorch autograd does. After this notebook, autograd will have no mystery.

---

## Key Terminology

| Term | Plain English | Origin / Context |
|---|---|---|
| **Gradient** | The direction and magnitude of steepest increase for a function. For a loss function, the negative gradient points toward lower loss. | Core calculus concept. In ML: `dL/dW` tells us how to change W to reduce loss |
| **Chain rule** | If `y = f(g(x))`, then `dy/dx = dy/dg · dg/dx`. Derivatives compose by multiplication. | Calculus 101 — but it's the entire engine behind backprop |
| **Backpropagation** | Algorithm to efficiently compute gradients for ALL parameters in one backward pass | Invented by Rumelhart, Hinton & Williams (1986). The key insight that made training deep networks practical |
| **Computational graph** | A DAG (directed acyclic graph) where nodes are operations and edges are data flow | Every deep learning framework (PyTorch, TensorFlow) builds this internally |
| **Autograd** | Automatic differentiation — the framework computes gradients for you | PyTorch's `loss.backward()` does exactly what we build manually here |
| **Gradient checking** | Verify analytical gradients against slow-but-guaranteed numerical gradients | Essential debugging tool. If they don't match, your backward pass has a bug |
| **Finite differences** | Approximate derivative: `f'(x) ≈ (f(x+h) - f(x-h)) / 2h` | The "brute force" gradient — slow but always correct |
| **Loss function** | Measures prediction error as a single number the optimizer minimizes | Different tasks use different losses: MSE for regression, cross-entropy for classification |
| **Cross-entropy loss** | `-log(P(correct))` — penalizes the model for being uncertain about the right answer | THE standard loss for language models and classification |
| **Softmax** | Converts raw scores (logits) into probabilities that sum to 1 | `softmax(x_i) = exp(x_i) / Σexp(x_j)`. Always applied before cross-entropy |
| **Logits** | Raw model output scores BEFORE softmax — can be any real number | Named because they're the input to the logistic (softmax) function |
| **Perplexity** | `exp(cross_entropy_loss)` — roughly "how many choices the model is confused between" | Perplexity 10 means the model is as uncertain as choosing randomly from 10 options |
| **Convergence** | When loss stops decreasing meaningfully — training is "done" | The model has learned as much as it can (or you've hit other limits) |
| **Divergence** | When loss increases or becomes NaN — training is broken | Usually caused by learning rate too high |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: The Chain Rule — How Gradients Flow

If `L = f(g(h(x)))`, the chain rule says:

```
dL/dx = dL/df · df/dg · dg/dh · dh/dx
```

In a neural network:
- Each **layer** is a function in this chain
- The **loss** is the outermost function
- Gradients flow **backward** from loss to input
- Each layer only needs to know: the gradient coming FROM above, and its own local gradient

```
Forward:  x → [Layer1] → h → [Layer2] → g → [Loss] → L
Backward: dL/dx ← [Layer1] ← dL/dh ← [Layer2] ← dL/dg ← [Loss] ← 1
```

## Part 2: Building Modular Layers with Backward

Each layer implements:
- `forward(x)` → save input, return output
- `backward(grad_output)` → compute gradients for parameters AND input, return `grad_input`

The key contract: `backward()` receives `dL/d(output)` and returns `dL/d(input)`.

In [ ]:
class Linear:
    """y = x @ W.T + b
    
    Backward:
        dL/dW = grad_output.T @ x     (how to update weights)
        dL/db = sum(grad_output)       (how to update biases)
        dL/dx = grad_output @ W        (what to pass to previous layer)
    """
    
    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / in_features)  # He initialization
        self.W = np.random.randn(out_features, in_features) * scale
        self.b = np.zeros(out_features)
        # Gradient storage
        self.grad_W = None
        self.grad_b = None
    
    def forward(self, x):
        self.x = x  # save for backward
        return x @ self.W.T + self.b
    
    def backward(self, grad_output):
        # grad_output shape: (batch, out_features)
        self.grad_W = grad_output.T @ self.x     # (out, in)
        self.grad_b = np.sum(grad_output, axis=0) # (out,)
        grad_input = grad_output @ self.W          # (batch, in)
        return grad_input
    
    def parameters(self):
        return [(self.W, self.grad_W, 'W'), (self.b, self.grad_b, 'b')]


class ReLU:
    """y = max(0, x)
    
    Backward: gradient passes through where x > 0, blocked where x <= 0.
    Think of it as a gate: open for positive, closed for negative.
    """
    
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)
    
    def backward(self, grad_output):
        return grad_output * (self.x > 0).astype(float)
    
    def parameters(self):
        return []  # no learnable parameters


class Sigmoid:
    """y = 1 / (1 + exp(-x))
    
    Backward: dy/dx = y * (1 - y)  — elegant and efficient!
    """
    
    def forward(self, x):
        self.output = 1.0 / (1.0 + np.exp(-x))
        return self.output
    
    def backward(self, grad_output):
        return grad_output * self.output * (1 - self.output)
    
    def parameters(self):
        return []


class MSELoss:
    """L = mean((prediction - target)^2)
    
    Backward: dL/d(pred) = 2 * (pred - target) / n
    """
    
    def forward(self, prediction, target):
        self.prediction = prediction
        self.target = target
        return np.mean((prediction - target) ** 2)
    
    def backward(self):
        n = self.prediction.shape[0]
        return 2.0 * (self.prediction - self.target) / n


print("All layer classes defined with forward() and backward() methods.")
print("Each backward() takes grad from above, returns grad for below.")

## Part 3: Sequential Model — Automatic Forward & Backward

Now we chain layers together. The magic:
- **Forward**: pass data through each layer in order
- **Backward**: pass gradients through each layer in **reverse** order

This is the same pattern as PyTorch's `nn.Sequential`.

In [ ]:
class Sequential:
    """Chain layers together: forward flows left-to-right, backward right-to-left."""
    
    def __init__(self, *layers):
        self.layers = layers
    
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad
    
    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params


class SGD:
    """Stochastic Gradient Descent: param -= lr * gradient
    
    The simplest optimizer. AdamW (used in real LLMs) is a fancy version of this
    with momentum and adaptive learning rates.
    """
    
    def __init__(self, model, lr=0.1):
        self.model = model
        self.lr = lr
    
    def step(self):
        for param, grad, name in self.model.parameters():
            param -= self.lr * grad  # in-place update
    
    def zero_grad(self):
        for param, grad, name in self.model.parameters():
            grad *= 0  # reset gradients


# Build the XOR network using our modular components
np.random.seed(42)
model = Sequential(
    Linear(2, 8),    # input -> hidden
    ReLU(),          # activation
    Linear(8, 1),    # hidden -> output
    Sigmoid()        # squash to (0,1)
)

loss_fn = MSELoss()
optimizer = SGD(model, lr=1.0)

print("Model architecture:")
for i, layer in enumerate(model.layers):
    print(f"  Layer {i}: {layer.__class__.__name__}")
print(f"\nTotal parameters: {sum(p.size for p, _, _ in model.parameters())}")

In [ ]:
# XOR data
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
y = np.array([[0], [1], [1], [0]], dtype=np.float64)

# Training loop — now using our modular components
losses = []
for epoch in range(5000):
    # Forward pass
    output = model.forward(X)
    loss = loss_fn.forward(output, y)
    losses.append(loss)
    
    # Backward pass — this is the key!
    # 1. Get initial gradient from loss
    grad = loss_fn.backward()
    # 2. Propagate backward through all layers (automatic via Sequential)
    model.backward(grad)
    # 3. Update weights
    optimizer.step()
    optimizer.zero_grad()
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | Loss: {loss:.6f}")

# Verify
predictions = model.forward(X)
print(f"\nFinal predictions: {predictions.ravel().round(3)}")
print(f"Expected:          [0, 1, 1, 0]")
print(f"Correct: {np.all(np.round(predictions) == y)}")

## Part 4: Gradient Checking — Verify Our Math

How do we know our backward pass is correct? **Numerical gradient checking.**

For any parameter `p`, the gradient can be approximated numerically:

```
dL/dp ≈ (L(p + h) - L(p - h)) / (2h)    where h is a tiny number (e.g., 1e-5)
```

This is slow (one forward pass per parameter) but **guaranteed correct** by calculus.
If our analytical gradient matches the numerical gradient, our backward pass is right.

In [ ]:
def numerical_gradient(model, loss_fn, X, y, param, h=1e-5):
    """Compute gradient numerically using finite differences.
    
    For each element in param, compute:
        dL/dp_i = (L(p_i + h) - L(p_i - h)) / (2*h)
    """
    grad = np.zeros_like(param)
    
    # Iterate over every element in the parameter
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        original = param[idx]
        
        # L(p + h)
        param[idx] = original + h
        output_plus = model.forward(X)
        loss_plus = loss_fn.forward(output_plus, y)
        
        # L(p - h)
        param[idx] = original - h
        output_minus = model.forward(X)
        loss_minus = loss_fn.forward(output_minus, y)
        
        # Numerical gradient
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        
        # Restore original value
        param[idx] = original
        it.iternext()
    
    return grad


# Build a fresh network for gradient checking
np.random.seed(123)
check_model = Sequential(
    Linear(2, 4),
    ReLU(),
    Linear(4, 1),
    Sigmoid()
)
check_loss = MSELoss()

# Forward + backward to get analytical gradients
output = check_model.forward(X)
loss = check_loss.forward(output, y)
grad = check_loss.backward()
check_model.backward(grad)

# Compare analytical vs numerical gradients for each parameter
print("Gradient Checking Results")
print("=" * 60)
print(f"{'Parameter':<15} {'Max Diff':>12} {'Status':>10}")
print("-" * 60)

all_ok = True
for param, analytical_grad, name in check_model.parameters():
    numerical_grad = numerical_gradient(check_model, check_loss, X, y, param)
    
    # Relative error (handles different scales)
    diff = np.max(np.abs(analytical_grad - numerical_grad))
    
    status = 'PASS' if diff < 1e-5 else 'FAIL'
    if status == 'FAIL':
        all_ok = False
    
    print(f"{name:<15} {diff:>12.2e} {status:>10}")

print("-" * 60)
print(f"Overall: {'ALL PASSED' if all_ok else 'SOME FAILED'}")

if all_ok:
    print("\nOur analytical gradients match the numerical gradients!")
    print("This proves our backward pass implementation is correct.")

## Part 5: Visualizing Gradient Flow

One of the key challenges in deep networks is **vanishing/exploding gradients**.

Let's visualize how gradient magnitudes change as they flow backward through layers.
This is why techniques like **RMSNorm** and **residual connections** (Phase 3) are critical.

In [ ]:
# Build a deeper network to see gradient flow
np.random.seed(42)
deep_model = Sequential(
    Linear(2, 16),
    ReLU(),
    Linear(16, 16),
    ReLU(),
    Linear(16, 16),
    ReLU(),
    Linear(16, 1),
    Sigmoid()
)
deep_loss = MSELoss()

# Forward + backward
output = deep_model.forward(X)
loss = deep_loss.forward(output, y)
grad = deep_loss.backward()
deep_model.backward(grad)

# Collect gradient magnitudes for each Linear layer
grad_magnitudes = []
layer_names = []
for i, layer in enumerate(deep_model.layers):
    if hasattr(layer, 'grad_W') and layer.grad_W is not None:
        mag = np.mean(np.abs(layer.grad_W))
        grad_magnitudes.append(mag)
        layer_names.append(f'Linear {len(grad_magnitudes)}')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(layer_names, grad_magnitudes, color=['#2196F3', '#4CAF50', '#FF9800', '#FF5722'])
ax.set_ylabel('Mean |Gradient|', fontsize=12)
ax.set_title('Gradient Magnitudes per Layer (Backward Direction →)', fontsize=13, fontweight='bold')
ax.set_xlabel('Layer (rightmost = closest to loss)', fontsize=11)

for bar, val in zip(bars, grad_magnitudes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
            f'{val:.2e}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

ratio = grad_magnitudes[0] / grad_magnitudes[-1] if grad_magnitudes[-1] > 0 else float('inf')
print(f"Gradient ratio (first layer / last layer): {ratio:.4f}")
print("If this ratio is very small → vanishing gradients (early layers learn slowly)")
print("If this ratio is very large → exploding gradients (training becomes unstable)")
print("\nModern solution: RMSNorm + Residual connections (covered in Notebook 04 & Phase 3)")

## Part 6: Cross-Entropy Loss (What LLMs Actually Use)

MSE works for XOR, but language models use **Cross-Entropy Loss**.

For a vocabulary of V words, the model outputs V probabilities (via softmax).
Cross-entropy measures how surprised the model is by the correct answer:

```
L = -log(P(correct_token))
```

If the model assigns probability 0.9 to the right word: `L = -log(0.9) = 0.105` (small loss)  
If the model assigns probability 0.01: `L = -log(0.01) = 4.6` (large loss)

This is the loss function we'll use in Phase 4 to train our transformer.

In [ ]:
def softmax(x):
    """Numerically stable softmax.
    
    Subtract max for numerical stability (prevents exp overflow).
    This is a critical trick — without it, exp(1000) = inf.
    """
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def cross_entropy_loss(logits, targets):
    """Cross-entropy loss: -log(softmax(logits)[target])
    
    logits: (batch, vocab_size) — raw model output before softmax
    targets: (batch,) — integer indices of correct tokens
    """
    probs = softmax(logits)
    batch_size = logits.shape[0]
    # Select probability of correct class for each sample
    correct_probs = probs[np.arange(batch_size), targets]
    return -np.mean(np.log(correct_probs + 1e-10))  # +eps for numerical stability


def cross_entropy_backward(logits, targets):
    """Gradient of cross-entropy loss w.r.t. logits.
    
    Beautiful result: dL/d(logits) = softmax(logits) - one_hot(targets)
    This simple formula is why cross-entropy + softmax is the standard combo.
    """
    probs = softmax(logits)
    batch_size = logits.shape[0]
    grad = probs.copy()
    grad[np.arange(batch_size), targets] -= 1  # subtract 1 at the correct class
    return grad / batch_size


# Demo: simulate a tiny vocabulary
vocab_size = 5
vocab = ['the', 'cat', 'sat', 'on', 'mat']

# Model outputs raw logits (before softmax)
logits = np.array([[2.0, 0.5, 0.1, 0.3, 0.1]])  # batch of 1
target = np.array([0])  # correct word is 'the' (index 0)

probs = softmax(logits)
loss = cross_entropy_loss(logits, target)
grad = cross_entropy_backward(logits, target)

print("Vocabulary:", vocab)
print(f"Correct word: '{vocab[target[0]]}' (index {target[0]})")
print(f"\nRaw logits:    {logits[0]}")
print(f"Softmax probs: {probs[0].round(4)}")
print(f"P(correct):    {probs[0, target[0]]:.4f}")
print(f"Loss:          {loss:.4f}")
print(f"\nGradient: {grad[0].round(4)}")
print(f"Notice: gradient is (prob - 1) for correct class, just prob for others")
print(f"This pushes the model to increase P(correct) and decrease P(others)")

In [ ]:
# Visualize how loss changes with confidence
confidences = np.linspace(0.01, 0.99, 100)
losses_ce = -np.log(confidences)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(confidences, losses_ce, color='#FF5722', linewidth=2)
ax.set_xlabel('P(correct token)', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('Cross-Entropy Loss vs Confidence in Correct Answer', fontsize=13, fontweight='bold')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax.annotate('50% confidence\nloss = 0.69', xy=(0.5, 0.69), xytext=(0.65, 2.0),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key insight for LLMs:")
print("- Loss of 4.0 means the model is basically guessing randomly")
print("- Loss of 2.0 means the model has some idea but isn't confident")
print("- Loss of 0.5 means the model is fairly confident and usually right")
print("- Perplexity = exp(loss), so loss 2.0 → perplexity ~7.4")

## Part 7: Training Dynamics — Learning Rate Matters

The learning rate controls how big each update step is:
- **Too high**: loss oscillates wildly or diverges
- **Too low**: learning is painfully slow
- **Just right**: smooth convergence

This is why modern LLMs use **warmup** (start small) then **decay** (gradually reduce).

In [ ]:
# Train XOR with different learning rates
learning_rates = [0.01, 0.1, 1.0, 5.0]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#FF5722']

fig, ax = plt.subplots(figsize=(10, 5))

for lr, color in zip(learning_rates, colors):
    np.random.seed(42)
    m = Sequential(Linear(2, 8), ReLU(), Linear(8, 1), Sigmoid())
    loss_fn_lr = MSELoss()
    opt = SGD(m, lr=lr)
    
    lr_losses = []
    for epoch in range(3000):
        out = m.forward(X)
        l = loss_fn_lr.forward(out, y)
        lr_losses.append(min(l, 1.0))  # clip for visualization
        g = loss_fn_lr.backward()
        m.backward(g)
        opt.step()
        opt.zero_grad()
    
    ax.plot(lr_losses, color=color, linewidth=1.5, label=f'lr={lr}')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Effect of Learning Rate on Training', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 0.5)
plt.tight_layout()
plt.show()

print("Observations:")
print("  lr=0.01: Too slow — barely learning after 3000 epochs")
print("  lr=0.1:  Reasonable but still slow for this problem")
print("  lr=1.0:  Good — converges quickly")
print("  lr=5.0:  Might be unstable (oscillations)")
print("\nIn Phase 4, we'll use lr=3e-4 with WSD schedule (warmup-stable-decay)")

## Summary & Key Takeaways

1. **Backpropagation** = chain rule applied to a computational graph
2. Each layer implements `forward()` and `backward()` — the same pattern PyTorch uses
3. **Gradient checking** verifies correctness: analytical gradient ≈ numerical gradient
4. **Cross-entropy + softmax** is the standard loss for language models (not MSE)
5. The gradient of cross-entropy is beautifully simple: `softmax(logits) - one_hot(target)`
6. **Learning rate** is the most important hyperparameter — too high = diverge, too low = slow

### Connection to PyTorch
What we built manually is exactly what PyTorch autograd does:
- `loss.backward()` = our `model.backward(loss_fn.backward())`
- `optimizer.step()` = our `optimizer.step()`
- `optimizer.zero_grad()` = our `optimizer.zero_grad()`

### What's Next
**Notebook 03 (Attention)** — the mechanism that makes transformers work:
- How does a model decide which words to "pay attention to"?
- What are Query, Key, Value?
- Why causal masking is essential for language models